# AIC Pipeline — executable walkthrough

Notebook này chạy đúng source hiện tại trong `src/aic_pipeline/` theo trình tự của `AIC_PROCESSING_ARCHITECTURE(2).md`. Chạy **Run All** từ trên xuống.

Đầu vào phải là `data/videos/Lxx_Vxx.mp4`. Pipeline chỉ ghi `Shot.json`, `Frame.json`, PNG và NPY vào `data/`.

In [1]:
# Cell 1 — import đúng source gốc
from pathlib import Path
import json
import os
import sys
import yaml

ROOT = Path('/home/long/Documents/AIC/BoldSearch')
os.chdir(ROOT)  # bảo đảm đường dẫn tương đối trong default.yaml giống CLI gốc
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from aic_pipeline.orchestrator import VideoPipelineOrchestrator
from aic_pipeline.services import (
    detect_shots, embed_frames, extract_frames, index_frames,
    preliminary_dedup_from_config, probe_video,
)
from aic_pipeline.storage import NpyFileVectorStore, atomic_json


In [2]:
# Cell 2 — cấu hình và kiểm tra video đầu vào
CONFIG_PATH = ROOT / 'configs' / 'default.yaml'
VIDEO_PATH = ROOT / 'data' / 'videos' / 'L21_V01.mp4'

pipeline = VideoPipelineOrchestrator.from_yaml(CONFIG_PATH)
config = pipeline.config
data_root = Path(config['paths']['data_root']).absolute()
video_path = VIDEO_PATH.absolute()  # giữ L21_V01 khi video là symlink
video_id = video_path.stem

assert video_path.is_file(), f'Không thấy video: {video_path}'
assert pipeline.VIDEO_ID_PATTERN.fullmatch(video_id), 'Tên video phải là Lxx_Vxx.mp4'
expected_video, shot_path, frame_json_path, frames_root = pipeline._paths(video_id)
assert video_path == expected_video.absolute()


## 1. Video probe → AutoShot → `Shot.json`

`detect_shots` dùng checkpoint AutoShot local. Chỉ trường `video_id` và `shots` được ghi vào `Shot.json`.

In [3]:
# Cell 3 — đúng phần AutoShot trong VideoPipelineOrchestrator.run
video_metadata = probe_video(video_path)
shots = detect_shots(video_id, video_path, video_metadata, config['autoshot'])
atomic_json(shot_path, {'video_id': video_id, 'shots': shots['shots']})


## 2. Index N frame → batch 10 → dedup sơ bộ → `Frame.json`

`frame_id` được tạo từ `frame_index` zero-pad. Batch chỉ để xử lý, không thay tên. Strategy mặc định `none` giữ mọi candidate.

In [4]:
# Cell 4 — index, batch và Frame.json ban đầu
frames = index_frames(
    video_id,
    shots,
    int(config['indexer']['sample_every_frames']),
    int(config['pipeline']['batch_size']),
    int(config['frame_id']['zero_pad_width']),
)
frames = preliminary_dedup_from_config(config['preliminary_dedup']).select(frames).frames
pipeline._save_frame_json(frame_json_path, video_id, video_path, frames)


## 3. Mapping: Video MP4 + `Frame.json` → PNG

Extractor seek từng `frame_index` và ghi `data/frames/<video_id>/<frame_id>.png`, sau đó cập nhật chính `Frame.json`.

In [5]:
# Cell 5 — mapping PNG
extract_frames(
    video_path,
    frames,
    frames_root,
    force_overwrite=bool(config.get('mapping', {}).get('overwrite_existing', False)),
)
pipeline._save_frame_json(frame_json_path, video_id, video_path, frames)


## 4. Embedding: PNG → NPY

Đổi `EMBEDDING_PROVIDER` thành `fgclip` để chạy model FGCLIP thật. `histogram` giúp thử toàn bộ luồng offline nhưng vẫn lưu NPY theo cùng quy tắc tên.

In [6]:
# Cell 6 — embedding và lưu data/vectors/<video_id>/<frame_id>.npy
EMBEDDING_PROVIDER = 'histogram'  # đổi thành 'fgclip' nếu model đã sẵn sàng
embedder, model_version = pipeline._embedder(EMBEDDING_PROVIDER)
vector_store = NpyFileVectorStore(data_root / 'vectors', model_version)
embed_frames(
    frames,
    embedder,
    vector_store,
    int(config['embedding']['batch_size']),
    force_overwrite=False,
)
pipeline._save_frame_json(frame_json_path, video_id, video_path, frames)


## 5. Similarity step = 2 → cập nhật `Frame.json`

Danh sách được sort theo `frame_index`, ghép từng cặp không chồng lấn. Nếu cosine similarity `> 0.4`, giữ frame phải và đánh dấu frame trái `DUPLICATE`.

In [7]:
# Cell 7 — similarity và cập nhật Frame.json cuối
pipeline._apply_similarity(frames, vector_store, float(config['similarity']['threshold']))
pipeline._save_frame_json(frame_json_path, video_id, video_path, frames)
pipeline.validate(video_id)


In [8]:
# Cell 8 — chỉ hiển thị kết quả trong notebook; không tạo file mới
frame_document = json.loads(frame_json_path.read_text(encoding='utf-8'))
{
    'video_id': video_id,
    'shot_count': len(shots['shots']),
    'frame_count': len(frame_document['frames']),
    'kept_count': sum(item['final_status'] == 'KEPT' for item in frame_document['frames']),
    'duplicate_count': sum(item['final_status'] == 'DUPLICATE' for item in frame_document['frames']),
    'shot_json': str(shot_path),
    'frame_json': str(frame_json_path),
}


{'video_id': 'L21_V01',
 'shot_count': 10,
 'frame_count': 30,
 'kept_count': 27,
 'duplicate_count': 3,
 'shot_json': 'data/metadata/L21_V01/Shot.json',
 'frame_json': 'data/metadata/L21_V01/Frame.json'}

## Xem source gốc đang được chạy

Cell bên dưới in chính xác method `run` trong source hiện tại. Notebook trên tách method này thành các cell để dễ quan sát trạng thái `shots` và `frames` giữa các giai đoạn.

In [9]:
# Cell tuỳ chọn — xem source canonical, không chạy lại pipeline
from inspect import getsource
print(getsource(VideoPipelineOrchestrator.run))


    def run(self, video_path: Path, embedding_provider: str | None = None) -> dict[str, Any]:
        # Keep the logical input name: resolving a symlink would replace
        # Lxx_Vxx.mp4 with the target's unrelated filename.
        video_path = video_path.absolute()
        if video_path.suffix.lower() != ".mp4":
            raise ValueError("only .mp4 input is supported")
        video_id = video_path.stem
        if not self.VIDEO_ID_PATTERN.fullmatch(video_id):
            raise ValueError("video filename must follow Lxx_Vxx.mp4, for example L21_V01.mp4")
        expected_video, shot_path, frame_path, frames_root = self._paths(video_id)
        if video_path != expected_video.absolute():
            raise ValueError(f"video must be stored at {expected_video}")

        metadata = probe_video(video_path)
        shots = detect_shots(video_id, video_path, metadata, self.config["autoshot"])
        atomic_json(shot_path, {"video_id": video_id, "shots": shots["shots"]})

        fram